In [2]:
import sys, pathlib, fitz
import re
import cleaning
import pandas as pd

In [3]:
pdf_file = "../../1-1 Review Standards/Standards/(content) BS_ISO-IEC_5338-2023.pdf"
doc = fitz.open(pdf_file) # open a document

In [4]:
print(doc[0].get_text("text"))

ISO/IEC 5338:2023(E)
Foreword ..........................................................................................................................................................................................................................................v
Introduction .............................................................................................................................................................................................................................. vi
1 
Scope .................................................................................................................................................................................................................................1
2 
Normative references .....................................................................................................................................................................................1
3 
Terms and definitions ...........................

In [5]:
def is_starting_process(s):
    return s.strip().isdigit()

def is_starting_process_v2(s):
    pattern = re.compile(r'^\d{1,2}\.\d{1,2}$')
    if pattern.search(s.strip()):
        return True
    else:
        return False

def is_starting_activity(s):
    pattern = re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}$')
    if pattern.search(s.strip()):
        return True
    else:
        return False
    
def is_starting_activity_and_name(s):
    pattern = re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}\n?.*?$')
    if pattern.search(s.strip()):
        return True
    else:
        return False

In [6]:
column_names = ["Process_ID", "Process", "Process_2_ID", "Process_2", "Activity_ID", "Activity"]
df = pd.DataFrame(columns=column_names)

procedure_pattern = r'^([\s\S]*?)\s?\.{2,}\s?\d+$'

reviewing_process_id = ""
reviewing_process_v2_id = ""
reviewing_activity = ""
hold_previuos_process = False
hold_previuos_process_v2 = False
hold_previuos_activity = False

reviewing_process_title = ""
reviewing_process_v2_title = ""

for page in doc:
    for line in page.get_text("text").split("\n"):
        line = line.strip()
        if hold_previuos_process:
            pattern = re.compile(procedure_pattern)
            matches = pattern.findall(line)
            #print(str(reviewing_process_id), " ",matches)
            reviewing_process_title = matches[0]
            hold_previuos_process = False

        if hold_previuos_process_v2:
            pattern = re.compile(procedure_pattern)
            matches = pattern.findall(line)
            #print(str(reviewing_process_v2_id), " ", matches[0])
            reviewing_process_v2_title = matches[0]
            hold_previuos_process_v2 = False

        if hold_previuos_activity:
            pattern = re.compile(procedure_pattern)
            matches = pattern.findall(line)
            new_row = pd.DataFrame([{"Process_ID":reviewing_process_id, "Process":reviewing_process_title.strip(), "Process_2_ID": reviewing_process_v2_id, "Process_2": reviewing_process_v2_title.strip(), "Activity_ID":reviewing_activity, "Activity":matches[0].strip()}])
            df = pd.concat([df, new_row], ignore_index=True)
            hold_previuos_activity = False

        if is_starting_process(line):
            if int(line)<6:
                continue 
            reviewing_process_id = line            
            hold_previuos_process = True
        else:
            hold_previuos_process = False

        if is_starting_process_v2(line):
            if int(line[0])<6:
                continue 
            reviewing_process_v2_id = line            
            hold_previuos_process_v2 = True
        else:
            hold_previuos_process_v2 = False

        if is_starting_activity_and_name(line):
            if int(line[0])<6:
                continue
            if is_starting_activity(line):
                reviewing_activity = line     
                hold_previuos_activity = True
            else:
                pattern = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\s+([aA-zZ\s]+).*?$')
                matches = pattern.findall(line)
                new_row = pd.DataFrame([{"Process_ID":reviewing_process_id, "Process":reviewing_process_title.strip(), "Process_2_ID": reviewing_process_v2_id, "Process_2": reviewing_process_v2_title.strip(), "Activity_ID":matches[0][0], "Activity":matches[0][1].strip()}])
                df = pd.concat([df, new_row], ignore_index=True)
                
        else:
            hold_previuos_activity = False

In [8]:
display(df.head(10))

,Process_ID,Process,Process_2_ID,Process_2,Activity_ID,Activity
0,6,AI System life cycle processes,6.1,Agreement processes,6.1.1,Acquisition process
1,6,AI System life cycle processes,6.1,Agreement processes,6.1.2,Supply process
2,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.1,Life cycle model management process
3,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.2,Infrastructure management process
4,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.3,Portfolio management process
5,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.4,Human resource management process
6,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.5,Quality management process
7,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.6,Knowledge management process
8,6,AI System life cycle processes,6.3,Technical management processes,6.3.1,Project planning process
9,6,AI System life cycle processes,6.3,Technical management processes,6.3.2,Project assessment and control process


In [9]:
pdf_file = "../../1-1 Review Standards/Standards/processes from 5338.pdf"
doc = fitz.open(pdf_file) # open a document

In [73]:
def remove_header(page):
    pattern = re.compile(r'^ISO/IEC 5338:2023\(E\)\n')
    return pattern.sub("", page.strip())
    
def remove_footer(page):
    pattern = r'© ISO\/IEC 2023 \– All rights reserved\s*?\n+?\d{1,2}\s*?\n+?BS ISO/IEC 5338:2023$'
    return re.sub(re.compile(pattern), "", page.strip())

def is_starting_process(s):
    s = s.strip()
    
    if s.isdigit():
        return True
    elif re.compile(r'^\d{1,2}\s[A-Za-z\s]+$').search(s):
        return True
    elif re.compile(r'^\d{1,2}\.\d{1,2}\s[A-Za-z\s]+$').search(s):
        return True
    elif re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}$').search(s):
        return True
    
def is_starting_activity_and_name_2(s):
    #pattern = re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}\s+?.*?$')
    #pattern = re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}\b(?!.\d)\b$')
    #pattern = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\b[^.]*$')
    re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}\s[A-Za-z\s]+')
    if pattern.search(s.strip()):
        return True
    else:
        return False
    
def return_activity_last_number(s):
    #pattern = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\s+?.*?$')
    #pattern = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\b[^.]*$')
    re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\s[A-Za-z\s]+')
    # Use re.match() to search for the pattern in the string
    match = re.match(pattern, s)

    # Check if a match is found
    if match:
        # Extract the number part from the matched string
        return int(match.group(1).split(".")[-1])
    else:
        return 0

def is_starting_subsection_with_name(s, label):
    #6.4.14.3 Activities and tasks
    pattern = re.compile(r'^\d{1,2}\.\d{1,2}\.\d{1,2}\.\d{1,2}\s'+ re.escape(label) + r'$')
    if pattern.search(s.strip()):
        #print(f"{s} element found")
        return True
    else:
        #print(f"{s} element not found")
        return False
    
def is_starting_subsection(s):
    pattern = re.compile(r'^\d{1,2}(?:\.\d{1,2}){3}$')
    if pattern.search(s.strip()):
        #print(f"{s} element found")
        return True
    else:
        #print(f"{s} element not found")
        return False
    
def check_text_in(df, text):
    
    # Flatten the DataFrame to a Series
    flattened_series = df.stack()
    
    # Check if the text is not present in any cell
    text_not_in_dataframe = text in flattened_series.values
    
    return text_not_in_dataframe

def is_empty_text(text):
    pattern = re.compile(r'\S')
    
    if pattern.search(text):
        return False
    else:
        return True

In [97]:
column_names = ["Process_ID", "Process_Name", "Process_2_ID", "Process_2_Name", "Activity_ID", "Activity_Name"]
df_5338 = pd.DataFrame(columns=column_names)

next_index_line = 0
next_index_activity = 0
tasks_per_activity = []
new_row = None

current_process_name = df.loc[0, "Process"]
current_process_id = df.loc[0, "Process_ID"]
current_process_2_name = df.loc[0, "Process_2"]
current_process_2_id = df.loc[0, "Process_2_ID"]
current_activity_name = df.loc[0, "Activity"]
current_activity_id = df.loc[0, "Activity_ID"]

new_row_activity = {}
next_activity_name = ""
next_activity_id = ""
next_process_2_name = ""
next_process_2_id = ""

posible_subtitles = ["Purpose", "Outcomes", "Outcomes, activities and tasks", "Activities and tasks", "AI-specific particularities"]
label_new_row_item = ""

second_pattern_found = False
follows_next_text = False

start_index = 0

#start looking for the section.   
for index_page in range(start_index, doc.page_count):
    # Remove the footer of the page. 
    page = doc[index_page].get_text("text")
    new_page = remove_header(page.strip())
    new_page = remove_footer(new_page)
    lines = new_page.strip().split("\n")
    
    for index_line in range(0, len(lines)):
        
        next_index_line = index_line + 1
        line = lines[index_line].strip()

        if line.strip() == "" or line.strip() == "\n":
            continue
        
        elif index_page == 0 and line in current_activity_name and current_activity_id in lines[next_index_line]:
            
            if (next_index_activity+1) < len(df):
                next_index_activity = next_index_activity+1
                next_activity_name = df.loc[next_index_activity, "Activity"]
                next_activity_id = df.loc[next_index_activity, "Activity_ID"]
                next_process_2_name = df.loc[next_index_activity, "Process_2"]
                next_process_2_id = df.loc[next_index_activity, "Process_2_ID"]

            new_row = None
            new_row_activity = {
                "Process_ID":current_process_id, 
                "Process_Name":current_process_name, 
                "Process_2_ID":current_process_2_id, 
                "Process_2_Name":current_process_2_name, 
                "Activity_ID":current_activity_id, 
                "Activity_Name":current_activity_name
                }
            
            follows_next_text = False

        elif line in next_activity_name and next_activity_id in lines[next_index_line]:
            
            current_activity_id = next_activity_id
            current_activity_name = next_activity_name
            current_process_2_id = next_process_2_id
            current_process_2_name = next_process_2_name
            
            if (next_index_activity+1) < len(df):
                next_index_activity = next_index_activity+1
                next_activity_name = df.loc[next_index_activity, "Activity"]
                next_activity_id = df.loc[next_index_activity, "Activity_ID"]
                next_process_2_name = df.loc[next_index_activity, "Process_2"]
                next_process_2_id = df.loc[next_index_activity, "Process_2_ID"]

            new_row = pd.DataFrame([new_row_activity])
            df_5338 = pd.concat([df_5338, new_row], ignore_index=True)

            new_row = None
            new_row_activity = {
                "Process_ID":current_process_id, 
                "Process_Name":current_process_name, 
                "Process_2_ID":current_process_2_id, 
                "Process_2_Name":current_process_2_name, 
                "Activity_ID":current_activity_id, 
                "Activity_Name":current_activity_name
                }
                        
            follows_next_text = False
                    
        elif index_page >= 17 and is_starting_activity_and_name_2(line) and return_activity_last_number(line)>9:
            
            new_row = pd.DataFrame([new_row_activity])
            df_5338 = pd.concat([df_5338, new_row], ignore_index=True)
                        
            pattern = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\s([^\d.]*)$')
            matches = pattern.findall(line)
        
            current_activity_id = matches[0][0]
            current_activity_name = matches[0][1]
            current_process_2_name = df.loc[next_index_activity, "Process_2"]
            current_process_2_id = df.loc[next_index_activity, "Process_2_ID"]
            next_index_activity = next_index_activity+1

            new_row = None
            new_row_activity = {
                "Process_ID":current_process_id, 
                "Process_Name":current_process_name, 
                "Process_2_ID":current_process_2_id, 
                "Process_2_Name":current_process_2_name, 
                "Activity_ID":current_activity_id, 
                "Activity_Name":current_activity_name
                }
                        
            next_activity_name = ""
            next_activity_id = ""
            next_process_2_name = ""
            next_process_2_id = ""         
            
            follows_next_text = False
            second_pattern_found = True

        elif second_pattern_found:            
            for subtitle in posible_subtitles:
                if is_starting_subsection_with_name(line, subtitle):
                    label_new_row_item = subtitle
                    new_row_activity[label_new_row_item] = ""
                    new_description = ""     
                    follows_next_text = True
                    break
            else:
                if follows_next_text:
                    new_row_activity[label_new_row_item] = new_row_activity[label_new_row_item] + " " + line
            
        else:
            if is_starting_subsection(line): 
                label_new_row_item = lines[index_line+1]
                new_row_activity[label_new_row_item] = ""
                new_description = ""     
                follows_next_text = True        

            elif is_starting_process(line.strip()) and check_text_in(df, lines[index_line+1]):
                follows_next_text = False
                continue
            
            elif follows_next_text and line not in posible_subtitles:
                new_row_activity[label_new_row_item] = new_row_activity[label_new_row_item]  + " " + line 
                
            elif line not in posible_subtitles:
                label_new_row_item = posible_subtitles[0] # default
                new_row_activity[label_new_row_item] = new_row_activity[label_new_row_item] +  " " + line  if label_new_row_item in new_row_activity else line

new_row = pd.DataFrame([new_row_activity])
df_5338 = pd.concat([df_5338, new_row], ignore_index=True)
            

In [99]:
df_5338.head()

,Process_ID,Process_Name,Process_2_ID,Process_2_Name,Activity_ID,Activity_Name,Purpose,"Outcomes, activities and tasks",AI-specific particularities,Outcomes,Activities and tasks
0,6,AI System life cycle processes,6.1,Agreement processes,6.1.1,Acquisition process,The purpose of the acquisition process is to ...,"The outcomes, activities and tasks provided i...",The acquisition process described in ISO/IEC/...,NaN,NaN
1,6,AI System life cycle processes,6.1,Agreement processes,6.1.2,Supply process,The purpose of the supply process is to provi...,"The outcomes, activities and tasks provided i...",There are no additional activities or tasks d...,NaN,NaN
2,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.1,Life cycle model management process,"The purpose, outcomes, activities and tasks pr...",NaN,NaN,NaN,NaN
3,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.2,Infrastructure management process,"The purpose, outcomes, activities and tasks pr...",NaN,NaN,NaN,NaN
4,6,AI System life cycle processes,6.2,Organizational project-enabling processes,6.2.3,Portfolio management process,The purpose of the portfolio management proce...,"The outcomes, activities and tasks provided i...",There are no additional activities or tasks d...,NaN,NaN


In [100]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = 'output-5338_v3.xlsx'

# Export the DataFrame to Excel
df_5338.to_excel(excel_file_path, index=False)